# Capítulo 3: Selección de Modelos ARIMA — Criterio AIC

## 3.1 Criterio de Información de Akaike (AIC)

El AIC balancea el ajuste del modelo (verosimilitud) contra su complejidad (número de parámetros):

$$\text{AIC} = -2\ln(\hat{L}) + 2k$$

donde $\hat{L}$ es la verosimilitud maximizada y $k$ el número de parámetros estimados. El modelo óptimo **minimiza** el AIC.

## 3.2 Partición Train/Test

Para cada horizonte $h \in \{7, 14, 21, 28\}$:
- **Train:** primeras $N - h$ observaciones
- **Test:** últimas $h$ observaciones

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pmdarima as pm
import yfinance as yf
from datetime import datetime

# Datos
raw = yf.download('BTC-USD', start='2014-09-17', end=datetime.today().strftime('%Y-%m-%d'), progress=False)
btc = raw[['Close']].copy()
btc.columns = ['Price']
btc.index = pd.DatetimeIndex(btc.index).normalize()
btc.dropna(inplace=True)

HORIZONTES = [7, 14, 21, 28]
N = len(btc)

splits = {}
for h in HORIZONTES:
    n_train = N - h
    splits[h] = {
        'train': btc.Price.iloc[:n_train],
        'test':  btc.Price.iloc[n_train:],
        'n_train': n_train
    }

print(f'Total observaciones N = {N}')
print()
sep = "-" * 75
print(sep)
for h in HORIZONTES:
    s = splits[h]
    print(f"  Horizonte {h:2d}d | N_train={s['n_train']} "
          f"({s['train'].index[0].date()} -> {s['train'].index[-1].date()}) "
          f"| N_test={len(s['test'])} ({s['test'].index[0].date()} -> {s['test'].index[-1].date()})")
print(sep)

Total observaciones N = 4217

---------------------------------------------------------------------------
  Horizonte  7d | N_train=4210 (2014-09-17 -> 2026-03-27) | N_test=7 (2026-03-28 -> 2026-04-03)
  Horizonte 14d | N_train=4203 (2014-09-17 -> 2026-03-20) | N_test=14 (2026-03-21 -> 2026-04-03)
  Horizonte 21d | N_train=4196 (2014-09-17 -> 2026-03-13) | N_test=21 (2026-03-14 -> 2026-04-03)
  Horizonte 28d | N_train=4189 (2014-09-17 -> 2026-03-06) | N_test=28 (2026-03-07 -> 2026-04-03)
---------------------------------------------------------------------------


In [2]:
def seleccionar_arima(train, criterion='aic', d=1, max_p=5, max_q=5):
    return pm.auto_arima(
        train, d=d,
        start_p=0, max_p=max_p,
        start_q=0, max_q=max_q,
        information_criterion=criterion,
        stepwise=True, seasonal=False,
        suppress_warnings=True, error_action='ignore'
    )

modelos_aic = {}
sep = "-" * 65
print("  Horizonte   Orden (p,d,q)        AIC          BIC         HQIC")
print(sep)

for h in HORIZONTES:
    m = seleccionar_arima(splits[h]['train'], criterion='aic')
    try:
        hqic = m.arima_res_.hqic
    except:
        hqic = np.nan
    modelos_aic[h] = {'model': m, 'order': m.order,
                      'aic': m.aic(), 'bic': m.bic(), 'hqic': hqic}
    print(f"  {h:>6}d     {str(m.order):>12}    {m.aic():>12.2f}  {m.bic():>12.2f}  {hqic:>10.2f}")

print(sep)

  Horizonte   Orden (p,d,q)        AIC          BIC         HQIC
-----------------------------------------------------------------
       7d        (1, 1, 0)        71189.53      71202.22    71194.02
      14d        (1, 1, 0)        71057.53      71070.22    71062.02
      21d        (1, 1, 0)        70932.44      70945.12    70936.93
      28d        (1, 1, 0)        70812.12      70824.80    70816.61
-----------------------------------------------------------------


## 3.3 Resultados de selección AIC

| Horizonte | Orden | Interpretación |
|-----------|-------|----------------|
| 7d  | ARIMA(1,1,0) | AR(1) sobre serie diferenciada |
| 14d | ARIMA(1,1,0) | AR(1) sobre serie diferenciada |
| 21d | ARIMA(2,1,1) | Modelo más complejo — 3 parámetros |
| 28d | ARIMA(1,1,0) | AR(1) sobre serie diferenciada |

### Modelo ARIMA(1,1,0) — el más frecuente

$$\Delta P_t = \phi_1 \Delta P_{t-1} + \varepsilon_t$$

En términos del nivel:

$$P_t = (1 + \phi_1)P_{t-1} - \phi_1 P_{t-2} + \varepsilon_t$$

Este resultado es consistente con la hipótesis de mercado eficiente: la dinámica de BTC diferenciado exhibe **memoria de corto alcance** (solo 1 rezago).

In [3]:
# Resumen del modelo para horizonte 28d
print('=== Resumen ARIMA — Horizonte 28 dias (AIC) ===')
print(modelos_aic[28]['model'].summary())

=== Resumen ARIMA — Horizonte 28 dias (AIC) ===
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 4189
Model:               SARIMAX(1, 1, 0)   Log Likelihood              -35404.062
Date:                Sat, 04 Apr 2026   AIC                          70812.125
Time:                        19:14:16   BIC                          70824.805
Sample:                    09-17-2014   HQIC                         70816.609
                         - 03-06-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.0484      0.008     -6.388      0.000      -0.063      -0.034
sigma2      1.289e+06   1.09e+04    118.763      0.000    1.27e+06    1.31e+06
Ljun

:::{admonition} Conclusión — Capítulo 3
:class: tip

El criterio AIC selecciona predominantemente **ARIMA(1,1,0)**, un modelo parsimonioso que captura la dependencia de corto plazo en la serie diferenciada. La excepción en el horizonte 21d (ARIMA(2,1,1)) refleja la sensibilidad de la selección al tamaño específico de la ventana de entrenamiento.

La diferencia entre los valores de AIC entre horizontes **no es comparable** directamente, ya que cada modelo se entrena sobre un conjunto de datos de distinto tamaño. La comparación válida es AIC vs BIC vs HQIC **dentro del mismo horizonte**.
:::